### Step 1: Import Libraries
Load required packages for file handling, data processing, logging, and OSINT enrichment.



In [1]:
from pathlib import Path          # file/folder paths (works on Windows/macOS)
import pandas as pd               # data loading + cleaning
import logging                    # run log (what happened + when)
import re                         # pattern matching and string manipulation
import unicodedata                # Unicode character properties and normalization
import sys                        # system-specific parameters and functions
import glob                       # file path pattern matching and expansion
import numpy as np                # numerical computing and array operations
import requests                   # HTTP calls for OSINT API lookups (IP geolocation, breach data)
import socket                     # DNS / hostname resolution
import ipaddress                  # IP address validation and classification

### Step 2: Setup Project Folder Structure
Configure paths and create all required directories for the pipeline.

In [2]:
# ===============================
# Project root -> data_cleaning_scripts (sibling of 'Completed cleaning scripts')
# ===============================
ROOT = Path.cwd().parent.parent / "data_cleaning_scripts"
ROOT.mkdir(parents=True, exist_ok=True)

# ===============================
# Logs setup (MUST come first)
# ===============================
LOGS_DIR = ROOT / "new meet_logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOGS_DIR / "setup.log"

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Starting folder setup process.")

# ===============================
# Dataset base folder
# ===============================
DATASET_DIR = ROOT / "new meet_dataset"
NEW_MEET_RAW_DIR = DATASET_DIR / "new meet_raw"

# ===============================
# Subfolders
# ===============================
RAW_DIR = NEW_MEET_RAW_DIR / "new meet_raw"
PROCESSED_DIR = NEW_MEET_RAW_DIR / "new meet_processed"
GARBAGE_DIR = NEW_MEET_RAW_DIR / "new meet_garbage"
INGESTED_DIR = NEW_MEET_RAW_DIR / "new meet_ingested"

# ===============================
# Folder creation function
# ===============================
def create_folders():
    folders = [
        RAW_DIR,
        PROCESSED_DIR,
        GARBAGE_DIR,
        INGESTED_DIR,
        LOGS_DIR,
    ]

    for folder in folders:
        folder.mkdir(parents=True, exist_ok=True)
        logging.info(f"Ensured folder exists: {folder}")

# ===============================
# Create the folders
# ===============================

print("ROOT      :", ROOT)
create_folders()
print("PROCESSED :", PROCESSED_DIR)
print("GARBAGE   :", GARBAGE_DIR)

ROOT      : \\bca-org-00\Users Folder\jalleyne\Desktop\data_cleaning_scripts
PROCESSED : \\bca-org-00\Users Folder\jalleyne\Desktop\data_cleaning_scripts\new meet_dataset\new meet_raw\new meet_processed
GARBAGE   : \\bca-org-00\Users Folder\jalleyne\Desktop\data_cleaning_scripts\new meet_dataset\new meet_raw\new meet_garbage


#### ===============================
#### Load the raw CSV for inspection
#### ===============================

#### ===============================
#### Load data (NO cleaning yet, just inspection)
#### New Meet CSV uses semicolon delimiter
#### ===============================


In [3]:
import time

# Resolve New Meet raw CSV path
candidate_paths = [
    RAW_DIR / "new_meet_raw.csv",
    DATASET_DIR / "new meet_raw" / "new meet_raw" / "new_meet_raw.csv",
    ROOT / "raw dataset" / "new_meet_raw.csv",
    ROOT.parent / "raw dataset" / "new_meet_raw.csv",
    ROOT.parent.parent / "raw dataset" / "new_meet_raw.csv",
]

def is_readable_file(path_obj: Path) -> bool:
    try:
        path_obj = Path(path_obj)
        if not path_obj.is_file():
            return False
        with path_obj.open("rb") as fh:
            fh.read(1)
        return True
    except (PermissionError, OSError):
        return False

CSV_FILE = None
for p in candidate_paths:
    if is_readable_file(p):
        CSV_FILE = p
        break

# Fallback search under nearby parent folders
if CSV_FILE is None:
    search_roots = [DATASET_DIR, ROOT, ROOT.parent, ROOT.parent.parent]
    for base in search_roots:
        try:
            if base.exists():
                for match in base.rglob("new_meet_raw.csv"):
                    if is_readable_file(match):
                        CSV_FILE = match
                        break
        except PermissionError:
            logging.warning(f"Skipped unreadable search root: {base}")

        if CSV_FILE is not None:
            break

if CSV_FILE is None:
    logging.error("CSV file not found or not readable in expected locations.")
    raise FileNotFoundError("Missing or unreadable file: new_meet_raw.csv")

print(f"CSV file found: {CSV_FILE}")
print(f"File size: {CSV_FILE.stat().st_size:,} bytes")
logging.info(f"CSV file found: {CSV_FILE}")

# Retry read for transient network-share locks
last_error = None
for attempt in range(1, 4):
    try:
        df_raw = pd.read_csv(
            CSV_FILE,
            sep=";",
            low_memory=False,
            encoding="utf-8",
            encoding_errors="ignore",
        )
        break
    except PermissionError as exc:
        last_error = exc
        logging.warning(f"Read attempt {attempt} failed with PermissionError: {exc}")
        if attempt < 3:
            print(f"Read attempt {attempt}/3 failed due to file lock. Retrying...")
            time.sleep(1.5 * attempt)
else:
    raise PermissionError(
        f"Unable to read CSV after 3 attempts: {CSV_FILE}. "
        "Close the file if it is open in another program and run this cell again."
    ) from last_error

# Basic info
print(f"\nShape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print("\nColumns:")
print(df_raw.columns.tolist())
print("\nFirst 10 rows:")
print(df_raw.head(10))

CSV file found: \\bca-org-00\Users Folder\jalleyne\Desktop\Protexxa\raw dataset\new_meet_raw.csv
File size: 236,243,700 bytes

Shape: 500776 rows x 97 columns

Columns:
['IN_VALID', 'IN_NUM', 'IN_PSEUDO', 'IN_PASSWORD', 'IN_SEXE', 'IN_AGE', 'IN_DEP', 'IN_VILLE', 'IN_TAILLE', 'IN_POIDS', 'IN_YEUX', 'IN_CHEVEUX', 'IN_ORIGINE', 'IN_ORIGINEC', 'IN_PROF', 'IN_PERS', 'IN_STYLE', 'IN_R1', 'IN_R2', 'IN_R3', 'IN_RECH', 'IN_DESC', 'IN_EMAIL', 'IN_EMAILNV', 'IN_DATECREE', 'IN_TIMECREE', 'IN_DATEPASS', 'IN_TIMEPASS', 'IN_PHOTO', 'IN_PHOTO_BKP', 'IN_MODIF', 'IN_ADMDIAL', 'IN_IP', 'IN_CONN', 'IN_PORT', 'IN_NOIR', 'IN_CGU', 'IN_DATEER', 'IN_PAYS', 'IN_STATUT', 'IN_ENFANTS', 'IN_ENFANTSDES', 'IN_FUMEUR', 'IN_OPTIN', 'IN_A1', 'IN_A2', 'IN_A3', 'IN_A4', 'IN_A5', 'IN_A6', 'IN_A7', 'IN_A8', 'IN_A9', 'IN_A10', 'in_spam', 'in_abond1', 'in_abond2', 'in_visu', 'in_mailo1', 'in_mailo2', 'in_mailo3', 'in_mailo4', 'in_motiv', 'in_att', 'in_netude', 'in_tcarac', 'in_ager1', 'in_ager2', 'in_datenais', 'in_facebook

Some datasets contain:

broken rows

extra separators

incomplete records

These can affect row counts. # verying total number of lines in the dataset

In [4]:
with open(CSV_FILE, "r", encoding="utf-8", errors="ignore") as f:
    line_count = sum(1 for line in f)

print("Total lines in file:", line_count)

Total lines in file: 603972


#### Step 4: Column Normalization Summary
We standardized headers by removing IN_/in_ prefixes and translating key French names to English.
This keeps columns consistent for analysis (for example, IN_PASSWORD -> PASSWORD and IN_PSEUDO -> USERNAME).
#### Normalize and translate New Meet column names
#### 1) Remove IN_/in_ prefix
#### 2) Convert known French names to English
#### 3) Keep names uppercase with underscores

In [5]:


original_columns = df_raw.columns.tolist()

french_to_english = {
    "SEXE": "GENDER",
    "PSEUDO": "USERNAME",
    "DEP": "DEPARTMENT",
    "VILLE": "CITY",
    "TAILLE": "HEIGHT",
    "POIDS": "WEIGHT",
    "YEUX": "EYES",
    "CHEVEUX": "HAIR",
    "ORIGINE": "ORIGIN",
    "ORIGINEC": "ORIGIN_CODE",
    "PROF": "PROFESSION",
    "PERS": "PERSONALITY",
    "RECH": "LOOKING_FOR",
    "DESC": "DESCRIPTION",
    "DATECREE": "DATE_CREATED",
    "TIMECREE": "TIME_CREATED",
    "DATEPASS": "DATE_UPDATED",
    "TIMEPASS": "TIME_UPDATED",
    "PAYS": "COUNTRY",
    "ENFANTS": "CHILDREN",
    "ENFANTSDES": "CHILDREN_DETAILS",
    "FUMEUR": "SMOKER",
    "PRENOM": "FIRST_NAME",
}


def normalize_column_name(col_name: str) -> str:
    name = str(col_name).strip()

    # Remove IN_ prefix (case-insensitive)
    name = re.sub(r"^in_", "", name, flags=re.IGNORECASE)

    # Remove accents so names are ASCII-safe
    name = unicodedata.normalize("NFKD", name)
    name = "".join(ch for ch in name if not unicodedata.combining(ch))

    # Standardize formatting before translation
    name = name.replace(" ", "_")
    name = name.upper()

    # Translate known French labels
    name = french_to_english.get(name, name)

    # Keep only letters, numbers, and underscores
    name = re.sub(r"[^A-Z0-9_]", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")

    return name


new_columns = []
rename_dict = {}

for col in original_columns:
    new_col = normalize_column_name(col)

    # Avoid duplicate names after normalization
    base_name = new_col
    suffix = 2
    while new_col in new_columns:
        new_col = f"{base_name}_{suffix}"
        suffix += 1

    new_columns.append(new_col)

    if col != new_col:
        rename_dict[col] = new_col

# Always show a visible preview (even when no changes)
print("Column normalization preview (first 30):")
print("=" * 80)
for old_name, new_name in list(zip(original_columns, new_columns))[:30]:
    status = "CHANGED" if old_name != new_name else "SAME"
    print(f"{status:7} | {old_name:25} -> {new_name}")

# Apply renamed columns
df_raw.columns = new_columns

print("\nChanged columns (sample):")
print("=" * 80)
for old_name, new_name in list(rename_dict.items())[:40]:
    print(f"{old_name:25} -> {new_name}")

print("\nTotal renamed columns:", len(rename_dict))
print("Total columns:", len(df_raw.columns))

if len(rename_dict) == 0:
    print("Columns already normalized in current DataFrame.")

print("Example checks:")
print("IN_PASSWORD ->", normalize_column_name("IN_PASSWORD"))
print("IN_PSEUDO   ->", normalize_column_name("IN_PSEUDO"))

Column normalization preview (first 30):
CHANGED | IN_VALID                  -> VALID
CHANGED | IN_NUM                    -> NUM
CHANGED | IN_PSEUDO                 -> USERNAME
CHANGED | IN_PASSWORD               -> PASSWORD
CHANGED | IN_SEXE                   -> GENDER
CHANGED | IN_AGE                    -> AGE
CHANGED | IN_DEP                    -> DEPARTMENT
CHANGED | IN_VILLE                  -> CITY
CHANGED | IN_TAILLE                 -> HEIGHT
CHANGED | IN_POIDS                  -> WEIGHT
CHANGED | IN_YEUX                   -> EYES
CHANGED | IN_CHEVEUX                -> HAIR
CHANGED | IN_ORIGINE                -> ORIGIN
CHANGED | IN_ORIGINEC               -> ORIGIN_CODE
CHANGED | IN_PROF                   -> PROFESSION
CHANGED | IN_PERS                   -> PERSONALITY
CHANGED | IN_STYLE                  -> STYLE
CHANGED | IN_R1                     -> R1
CHANGED | IN_R2                     -> R2
CHANGED | IN_R3                     -> R3
CHANGED | IN_RECH                   -> LOOK

#### Step 5: Select Important Columns by Index
List all columns with index numbers and generate keep/drop/review suggestions so you can keep columns by index.
#### Requested: Username, first name, age, city, email, password
#### EMAILNV, COUNTRY, COUNTRY_2, and LASTNAME are exported to garbage and excluded from keep columns

In [6]:
indexed_columns = list(enumerate(df_raw.columns))
col_index_map = {str(col).upper(): idx for idx, col in indexed_columns}

# Keep only required analysis columns (FIRST_NAME placed next to USERNAME)
requested_candidates = {
    "USERNAME": ["USERNAME"],
    "FIRST_NAME": ["FIRST_NAME", "FIRSTNAME"],
    "AGE": ["AGE"],
    "CITY": ["CITY"],
    "EMAIL": ["EMAIL"],
    "PASSWORD": ["PASSWORD"],
}

resolved = {}
missing = []

for label, candidates in requested_candidates.items():
    found = None
    for candidate in candidates:
        if candidate in col_index_map:
            found = candidate
            break

    if found is None:
        missing.append(label)
    else:
        resolved[label] = found

# Build keep indexes in the same order as requested
keep_columns = [resolved[k] for k in requested_candidates.keys() if k in resolved]
keep_idx = [col_index_map[c] for c in keep_columns]

print("Requested keep columns with index numbers:")
print("=" * 80)
for req_label in requested_candidates.keys():
    if req_label in resolved:
        actual_col = resolved[req_label]
        print(f"{col_index_map[actual_col]:3} | {req_label:10} -> {actual_col}")
    else:
        print(f"--- | {req_label:10} -> NOT FOUND")

if missing:
    print("\nMissing requested items:", missing)

# Keep selected columns by index
df_keep = df_raw.iloc[:, keep_idx].copy()

print("\nSelected index list:", keep_idx)
print("Keep-only DataFrame shape:", df_keep.shape)
print("Kept columns:")
print(df_keep.columns.tolist())
print("\nPreview:")
print(df_keep.head(10))

# Explicitly export user-requested dropped columns to garbage
explicit_drop_cols = [c for c in ["EMAILNV", "COUNTRY", "COUNTRY_2", "LASTNAME"] if c in df_raw.columns]
if explicit_drop_cols:
    explicit_drop_file = GARBAGE_DIR / "explicit_dropped_columns.csv"
    df_raw[explicit_drop_cols].to_csv(explicit_drop_file, index=False, encoding="utf-8")
    print("\nExplicit dropped columns exported:", explicit_drop_cols)
    print("Explicit dropped columns file:", explicit_drop_file)

# Export all non-kept columns to dropped-columns garbage folder
dropped_columns = [col for col in df_raw.columns if col not in keep_columns]
df_dropped = df_raw[dropped_columns].copy()

garbage_export_dir = GARBAGE_DIR / "dropped columns"
garbage_export_dir.mkdir(parents=True, exist_ok=True)

dropped_file = garbage_export_dir / "dropped_columns.csv"
df_dropped.to_csv(dropped_file, index=False, encoding="utf-8")

print("\nDropped column count:", len(dropped_columns))
print("Dropped DataFrame shape:", df_dropped.shape)
print("Garbage folder:", garbage_export_dir)
print("Dropped columns file:", dropped_file.name)

Requested keep columns with index numbers:
  2 | USERNAME   -> USERNAME
 86 | FIRST_NAME -> FIRST_NAME
  5 | AGE        -> AGE
  7 | CITY       -> CITY
 22 | EMAIL      -> EMAIL
  3 | PASSWORD   -> PASSWORD

Selected index list: [2, 86, 5, 7, 22, 3]
Keep-only DataFrame shape: (500776, 6)
Kept columns:
['USERNAME', 'FIRST_NAME', 'AGE', 'CITY', 'EMAIL', 'PASSWORD']

Preview:
       USERNAME FIRST_NAME  AGE   CITY                      EMAIL   PASSWORD
0        Phil67   phillipe   23  Valff   oxygenix@netcourrier.com  tulipesev
1        Nono50        NaN   47    NaN    !chesnaye50@hotmail.com       1965
2  Jean jacques        NaN   60   Mons    jj_delannoy@hotmail.com       azde
3       Bridge1        NaN   42    NaN       byly1970@hotmail.com   70807080
4     Vincent59        NaN   42    NaN        jvg1@libertysurf.fr  DOMINIQUE
5        Valege   gilberte   57    NaN         ggdel94@hotmail.fr     Valege
6         Renea        NaN   65    NaN        gerard.a@wanadoo.fr       8779
7       

### Pre-Processing and Pre-Analysis
Perform data quality checks, identify missing values, duplicates, and basic statistics.


In [7]:
# Pre-processing + Data Quality Analysis (based on Step 5 keep columns)
print("PRE-PROCESSING + DATA QUALITY ANALYSIS - FILTERED DATASET")
print("=" * 70)

if "df_keep" not in globals():
    raise NameError("df_keep not found. Run Step 5 (column selection) first.")

# Start from kept columns
df_filtered = df_keep.copy()

# Basic preprocessing
# 1) Normalize blank/whitespace-only values to NaN
# 2) Trim text fields
# 3) Cast AGE to numeric when available
df_filtered = df_filtered.replace(r"^\s*$", np.nan, regex=True)

text_cols = df_filtered.select_dtypes(include=["object"]).columns
for col in text_cols:
    df_filtered[col] = df_filtered[col].astype(str).str.strip()
    df_filtered[col] = df_filtered[col].replace({"": np.nan, "nan": np.nan, "None": np.nan})

if "AGE" in df_filtered.columns:
    df_filtered["AGE"] = pd.to_numeric(df_filtered["AGE"], errors="coerce")

# 1. Missing values
print("\n1) MISSING VALUES")
missing_counts = df_filtered.isnull().sum()
missing_pcts = ((missing_counts / len(df_filtered)) * 100).round(2)
missing_df = pd.DataFrame(
    {
        "Column": missing_counts.index,
        "Missing_Count": missing_counts.values,
        "Missing_Percent": missing_pcts.values,
    }
).sort_values(["Missing_Count", "Missing_Percent"], ascending=False)
print(missing_df)

# 2. Duplicate rows
print("\n" + "=" * 70)
print("2) DUPLICATE RECORDS")
duplicate_count = df_filtered.duplicated().sum()
print(f"Total duplicates: {duplicate_count}")
if len(df_filtered) > 0:
    print(f"Duplicate percent: {(duplicate_count / len(df_filtered)) * 100:.2f}%")

# 3. Data types
print("\n" + "=" * 70)
print("3) DATA TYPES")
print(df_filtered.dtypes)

# 4. Basic statistics
print("\n" + "=" * 70)
print("4) BASIC STATISTICS")
print(df_filtered.describe(include="all").transpose())

# 5. Memory usage
print("\n" + "=" * 70)
print("5) MEMORY USAGE")
memory_usage = df_filtered.memory_usage(deep=True)
print(memory_usage)
print(f"Total memory (MB): {memory_usage.sum() / (1024 * 1024):.2f}")

# Save processed outputs
processed_file = PROCESSED_DIR / "new_meet_filtered.csv"
missing_report_file = PROCESSED_DIR / "new_meet_missing_report.csv"

df_filtered.to_csv(processed_file, index=False, encoding="utf-8")
missing_df.to_csv(missing_report_file, index=False, encoding="utf-8")

# Log summary
total_missing = int(missing_counts.sum())
logging.info(
    f"Pre-analysis complete: rows={len(df_filtered)}, "
    f"cols={df_filtered.shape[1]}, missing={total_missing}, duplicates={duplicate_count}"
)

print("\nSaved filtered dataset to:", processed_file)
print("Saved missing report to:", missing_report_file)
print("\nPre-analysis complete.")

PRE-PROCESSING + DATA QUALITY ANALYSIS - FILTERED DATASET

1) MISSING VALUES
       Column  Missing_Count  Missing_Percent
1  FIRST_NAME         267265            53.37
3        CITY         142167            28.39
4       EMAIL             12             0.00
5    PASSWORD              9             0.00
0    USERNAME              2             0.00
2         AGE              0             0.00

2) DUPLICATE RECORDS
Total duplicates: 372
Duplicate percent: 0.07%

3) DATA TYPES
USERNAME      object
FIRST_NAME    object
AGE            int64
CITY          object
EMAIL         object
PASSWORD      object
dtype: object

4) BASIC STATISTICS
               count  unique                                top   freq  \
USERNAME      500774  499459                     Elome.alain@ya     10   
FIRST_NAME    233511   63581                              karim   1572   
AGE         500776.0     NaN                                NaN    NaN   
CITY          358609   29573                            abid

#### Drop duplicate rows to garbage folder and keep unique rows

In [8]:

if "df_filtered" in globals():
    base_df = df_filtered.copy()
elif "df_keep" in globals():
    base_df = df_keep.copy()
else:
    raise NameError("df_filtered/df_keep not found. Run Step 5 first.")

duplicate_mask = base_df.duplicated(keep="first")
df_duplicates = base_df[duplicate_mask].copy()
df_unique = base_df[~duplicate_mask].copy()

# Save directly in main garbage folder
dupe_garbage_dir = GARBAGE_DIR
dupe_garbage_dir.mkdir(parents=True, exist_ok=True)

duplicates_file = dupe_garbage_dir / "duplicate_rows.csv"
df_duplicates.to_csv(duplicates_file, index=False, encoding="utf-8")

# Keep unique rows for downstream analysis
df_filtered = df_unique.copy()

print("Duplicate rows moved to garbage:", len(df_duplicates))
print("Unique rows kept:", len(df_filtered))
print("Duplicate file:", duplicates_file)

Duplicate rows moved to garbage: 372
Unique rows kept: 500404
Duplicate file: \\bca-org-00\Users Folder\jalleyne\Desktop\data_cleaning_scripts\new meet_dataset\new meet_raw\new meet_garbage\duplicate_rows.csv


### Implement Data Cleaning Steps
Apply logical, explainable cleaning transformations: remove duplicates, handle missing values, validate formats, and standardize data fields.



#### Drop LASTNAME, EMAILNV, COUNTRY, and COUNTRY_2 columns
All dropped columns are exported to a garbage file before removal.


In [9]:
# Drop columns not needed for downstream analysis
columns_to_drop = [
    c for c in ["LASTNAME", "EMAILNV", "COUNTRY", "COUNTRY_2"]
    if c in df_filtered.columns
]

if columns_to_drop:
    # Send all dropped-column data to garbage for traceability
    dropped_columns_file = GARBAGE_DIR / "dropped_columns_cleaning_step.csv"
    df_filtered[columns_to_drop].to_csv(dropped_columns_file, index=False, encoding="utf-8")

    df_filtered = df_filtered.drop(columns=columns_to_drop)
    print(f"Dropped columns: {columns_to_drop}")
    print(f"Dropped columns saved to garbage: {dropped_columns_file}")
else:
    print("No target columns found to drop (LASTNAME, EMAILNV, COUNTRY, COUNTRY_2).")

# Keep df_clean aligned if it already exists
if "df_clean" in globals():
    df_clean = df_filtered.copy()

print(f"Remaining columns ({len(df_filtered.columns)}): {list(df_filtered.columns)}")
print(f"Shape: {df_filtered.shape}")

No target columns found to drop (LASTNAME, EMAILNV, COUNTRY, COUNTRY_2).
Remaining columns (6): ['USERNAME', 'FIRST_NAME', 'AGE', 'CITY', 'EMAIL', 'PASSWORD']
Shape: (500404, 6)


#### AGE Cleaning
Remove rows with AGE outside the valid range of 13–100 (non-null). Invalid rows are exported to garbage.

In [10]:

# Valid age range for a dating site: 13 – 100
# Rows with a non-null AGE outside this range are moved to garbage.
AGE_MIN, AGE_MAX = 13, 100

if "AGE" in df_filtered.columns:
    non_null_mask    = df_filtered["AGE"].notna()
    invalid_age_mask = non_null_mask & (
        (df_filtered["AGE"] < AGE_MIN) | (df_filtered["AGE"] > AGE_MAX)
    )

    df_invalid_age = df_filtered[invalid_age_mask].copy()
    df_filtered    = df_filtered[~invalid_age_mask].copy()

    invalid_age_file = GARBAGE_DIR / "invalid_age_rows.csv"
    df_invalid_age.to_csv(invalid_age_file, index=False, encoding="utf-8")

    print(f"Invalid AGE rows moved to garbage : {len(df_invalid_age)}")
    print(f"Rows remaining                    : {len(df_filtered)}")
    print(f"Garbage file                      : {invalid_age_file}")
    print(f"AGE range in clean data           : {df_filtered['AGE'].min():.0f} – {df_filtered['AGE'].max():.0f}")
else:
    print("AGE column not found.")


Invalid AGE rows moved to garbage : 246
Rows remaining                    : 500158
Garbage file                      : \\bca-org-00\Users Folder\jalleyne\Desktop\data_cleaning_scripts\new meet_dataset\new meet_raw\new meet_garbage\invalid_age_rows.csv
AGE range in clean data           : 13 – 100


#### Email Validation
Rows where EMAIL is missing or does not match a valid email format are exported to garbage.
Only valid email-format rows are kept for downstream steps.

In [11]:
# Step 5: Validate email format
print("STEP 5: Validate Email Format")
print("=" * 60)

# Use df_clean in this step; fall back to df_filtered if needed
if "df_clean" not in globals():
    if "df_filtered" in globals():
        df_clean = df_filtered.copy()
    else:
        raise NameError("df_clean/df_filtered not found. Run previous cleaning steps first.")

# Detect email column name (supports both script style and normalized dataset)
if "email" in df_clean.columns:
    email_col = "email"
elif "EMAIL" in df_clean.columns:
    email_col = "EMAIL"
else:
    raise KeyError("Email column not found (expected 'email' or 'EMAIL').")

# Normalize email text
email_series = df_clean[email_col].astype(str).str.strip()

# Correct-format check: basic email structure user@domain.tld
email_pattern = r"^[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}$"
invalid_mask = ~email_series.str.fullmatch(email_pattern, na=False)

# Export invalid emails to garbage folder, then drop from clean data
df_invalid_email = df_clean[invalid_mask].copy()
invalid_emails = len(df_invalid_email)

invalid_email_file = GARBAGE_DIR / "invalid_email_rows.csv"
df_invalid_email.to_csv(invalid_email_file, index=False, encoding="utf-8")

df_clean = df_clean[~invalid_mask].copy()

# Keep pipeline variable in sync
if "df_filtered" in globals():
    df_filtered = df_clean.copy()

print("Validated email addresses")
print(f"Found: {invalid_emails} rows with invalid email format")
print(f"Invalid email rows moved to garbage: {invalid_email_file}")
print(f"Rows remaining in clean dataset: {len(df_clean)}")
logging.info(f"Step 5 - Validated email format ({invalid_emails} invalid)")

STEP 5: Validate Email Format
Validated email addresses
Found: 84267 rows with invalid email format
Invalid email rows moved to garbage: \\bca-org-00\Users Folder\jalleyne\Desktop\data_cleaning_scripts\new meet_dataset\new meet_raw\new meet_garbage\invalid_email_rows.csv
Rows remaining in clean dataset: 415891


#### Text Field Standardization
Apply title case to CITY, FIRST_NAME, COUNTRY, COUNTRY_2. Strip whitespace from USERNAME. NaN values are preserved.
Drop rows where FIR_NAME/FIRST_NAME is missing (null/blank placeholders).

In [12]:
NULL_PLACEHOLDERS = {"nan", "none", "", "n/a", "na", "null"}

# Title-case these columns and re-null any placeholder strings
title_case_cols = [c for c in ["CITY", "FIRST_NAME", "COUNTRY", "COUNTRY_2"] if c in df_filtered.columns]
for col in title_case_cols:
    df_filtered[col] = (
        df_filtered[col]
        .astype(str)
        .str.strip()
        .str.title()
    )
    df_filtered[col] = df_filtered[col].where(
        ~df_filtered[col].str.lower().isin(NULL_PLACEHOLDERS), other=np.nan
    )

# USERNAME: strip only - preserve original casing
if "USERNAME" in df_filtered.columns:
    df_filtered["USERNAME"] = df_filtered["USERNAME"].astype(str).str.strip()
    df_filtered["USERNAME"] = df_filtered["USERNAME"].where(
        ~df_filtered["USERNAME"].str.lower().isin(NULL_PLACEHOLDERS), other=np.nan
    )

# Drop rows where first-name is missing (supports FIR_NAME typo and FIRST_NAME variants)
first_name_candidates = ["FIR_NAME", "FIRST_NAME", "FIRSTNAME", "fir_name", "first_name", "firstname"]
first_name_col = next((c for c in first_name_candidates if c in df_filtered.columns), None)

if first_name_col is None:
    print("First-name column not found (expected FIR_NAME or FIRST_NAME). No rows dropped.")
else:
    before_rows = len(df_filtered)
    first_name_clean = df_filtered[first_name_col].astype(str).str.strip()
    missing_name_mask = (
        df_filtered[first_name_col].isna()
        | first_name_clean.str.lower().isin(NULL_PLACEHOLDERS)
    )

    dropped_missing_names = int(missing_name_mask.sum())
    if dropped_missing_names > 0:
        missing_name_file = GARBAGE_DIR / "missing_first_name_rows.csv"
        df_filtered.loc[missing_name_mask].to_csv(missing_name_file, index=False, encoding="utf-8")
        df_filtered = df_filtered.loc[~missing_name_mask].copy()
        print(f"Dropped rows missing {first_name_col}: {dropped_missing_names}")
        print(f"Dropped rows file: {missing_name_file}")
    else:
        print(f"No missing values found in {first_name_col}; no rows dropped.")

    print(f"Rows before: {before_rows}")
    print(f"Rows after : {len(df_filtered)}")

# Keep df_clean aligned if it already exists
if "df_clean" in globals():
    df_clean = df_filtered.copy()

print("Text fields standardized.")
print(f"\nSample values after standardization:")
sample_cols = title_case_cols + (["USERNAME"] if "USERNAME" in df_filtered.columns else [])
print(df_filtered[sample_cols].dropna(how="all").head(10).to_string(index=False))

Dropped rows missing FIRST_NAME: 222188
Dropped rows file: \\bca-org-00\Users Folder\jalleyne\Desktop\data_cleaning_scripts\new meet_dataset\new meet_raw\new meet_garbage\missing_first_name_rows.csv
Rows before: 415891
Rows after : 193703
Text fields standardized.

Sample values after standardization:
 CITY  FIRST_NAME  USERNAME
Valff    Phillipe    Phil67
  NaN    Gilberte    Valege
  NaN Jean Michel   James22
  NaN        Amel Ammoula78
  NaN      Franck Frankie34
  NaN        Eric      Pipi
  NaN      Yvette   Yvetta7
  NaN        Jluc   Boggy69
  NaN       Alain  Bouquett
Alger       Nanou  Nacera71


### Save Cleaned Dataset
Export the final cleaned DataFrame to the processed folder and log a summary.

In [13]:
# Keep output column order consistent
preferred_order = ["USERNAME", "FIRST_NAME", "AGE", "CITY", "EMAIL", "PASSWORD"]
ordered_existing = [c for c in preferred_order if c in df_filtered.columns]
remaining_cols = [c for c in df_filtered.columns if c not in ordered_existing]
df_filtered = df_filtered[ordered_existing + remaining_cols].copy()

cleaned_file = PROCESSED_DIR / "new_meet_cleaned.csv"
df_filtered.to_csv(cleaned_file, index=False, encoding="utf-8")

print("=" * 60)
print("CLEANED DATASET SUMMARY")
print("=" * 60)
print(f"Shape          : {df_filtered.shape}")
print(f"Columns        : {list(df_filtered.columns)}")
print(f"Saved to       : {cleaned_file}")
print()
print("Missing values per column:")
missing_final = df_filtered.isnull().sum()
missing_final_pct = ((missing_final / len(df_filtered)) * 100).round(2)
print(pd.DataFrame({"Missing": missing_final, "Pct": missing_final_pct}).to_string())

logging.info(
    f"Cleaned dataset saved: rows={len(df_filtered)}, "
    f"cols={df_filtered.shape[1]}, path={cleaned_file}"
)
print("\nCleaning pipeline complete.")

CLEANED DATASET SUMMARY
Shape          : (193703, 6)
Columns        : ['USERNAME', 'FIRST_NAME', 'AGE', 'CITY', 'EMAIL', 'PASSWORD']
Saved to       : \\bca-org-00\Users Folder\jalleyne\Desktop\data_cleaning_scripts\new meet_dataset\new meet_raw\new meet_processed\new_meet_cleaned.csv

Missing values per column:
            Missing    Pct
USERNAME          2   0.00
FIRST_NAME        0   0.00
AGE               0   0.00
CITY          60363  31.16
EMAIL             0   0.00
PASSWORD          2   0.00

Cleaning pipeline complete.


In [14]:
import pandas as pd

# Load your cleaned dataset
df = pd.read_csv(r"\\BCA-ORG-00\Users Folder\jalleyne\Desktop\Protexxa\data_cleaning_scripts\new meet_dataset\new meet_raw\new meet_processed\new_meet_cleaned.csv")

# Handle blanks as missing
city = df["CITY"].astype(str).str.strip().replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

total_rows = len(city)
missing_city = city.isna().sum()
relevant_city = total_rows - missing_city

missing_pct = (missing_city / total_rows) * 100
relevant_pct = (relevant_city / total_rows) * 100

print(f"Total rows: {total_rows}")
print(f"Missing CITY rows: {missing_city} ({missing_pct:.2f}%)")
print(f"Relevant CITY rows: {relevant_city} ({relevant_pct:.2f}%)")

Total rows: 193703
Missing CITY rows: 60363 (31.16%)
Relevant CITY rows: 133340 (68.84%)


#### Drop rows with missing CITY values and export them to garbage
#### Keep only rows with a valid CITY
#### Keep pipeline variable aligned

In [15]:

if "CITY" not in df_filtered.columns:
    raise KeyError("CITY column not found in df_filtered.")

city_clean = df_filtered["CITY"].astype(str).str.strip()
missing_city_mask = (
    df_filtered["CITY"].isna()
    | city_clean.str.lower().isin(["", "nan", "none", "null", "n/a", "na"])
)

df_missing_city = df_filtered.loc[missing_city_mask].copy()
missing_city_file = GARBAGE_DIR / "missing_city_rows.csv"
df_missing_city.to_csv(missing_city_file, index=False, encoding="utf-8")


df_filtered = df_filtered.loc[~missing_city_mask].copy()


if "df_clean" in globals():
    df_clean = df_filtered.copy()

print(f"Missing CITY rows moved to garbage: {len(df_missing_city)}")
print(f"Garbage file: {missing_city_file}")
print(f"Rows remaining: {len(df_filtered)}")

Missing CITY rows moved to garbage: 60363
Garbage file: \\bca-org-00\Users Folder\jalleyne\Desktop\data_cleaning_scripts\new meet_dataset\new meet_raw\new meet_garbage\missing_city_rows.csv
Rows remaining: 133340


### Cleaned Data Preview
Display the first 20 rows of the final cleaned dataset.

In [18]:

print("Shape:", df_filtered.shape)
print("=" * 60)
df_filtered.head(20)


Shape: (133340, 6)


,USERNAME,FIRST_NAME,AGE,CITY,EMAIL,PASSWORD
0,Phil67,Phillipe,23,Valff,oxygenix@netcourrier.com,tulipesev
34,Nacera71,Nanou,41,Alger,ceranota1@yahoo.fr,rayane
35,Cleopatre seul,Annick,33,Douala,kamsi394@yahoo.fr,30121977
48,Hafid2312sdf,Fgfg,44,Alger,oborf@getairmail.com,vcxvcxvcxvcvcx
49,Koffimarie,Marie-France,34,Abidjan,koffimarie@outlook.fr,amour@@@
60,C?c,Celeste,20,Brazzaville,balebanaastel@yahoo.fr,balebana1992
61,Bleckdash,Edith,36,Douala,edithdjounang@yahoo.fr,01051977
71,Youcef27,Youcef,30,Tizi-Ouzou,louseau2000@yahoo.fr,198200
76,Philippe-ulyss,Philippe,59,Coussey,mercenaire-9@hotmail.fr,brigantine
82,Haskelle13,Haskelle,34,Libreville,haskelle2013@yahoo.fr,gabon13


### Post-Cleaning Analysis
Evaluate final data quality after cleaning: completeness, duplicates, AGE profile, CITY coverage, and top email domains.

In [17]:
if "df_filtered" in globals():
    df_post = df_filtered.copy()
else:
    fallback_cleaned = PROCESSED_DIR / "new_meet_cleaned.csv"
    if not fallback_cleaned.exists():
        raise FileNotFoundError(f"Missing cleaned file: {fallback_cleaned}")
    df_post = pd.read_csv(fallback_cleaned)

placeholder_tokens = {"", "nan", "none", "null", "n/a", "na"}
df_post_norm = df_post.copy()

for col in df_post_norm.select_dtypes(include=["object"]).columns:
    s = df_post_norm[col].astype(str).str.strip()
    df_post_norm[col] = s.mask(s.str.lower().isin(placeholder_tokens), np.nan)

rows, cols = df_post_norm.shape
duplicate_rows = int(df_post_norm.duplicated().sum())

missing_counts = df_post_norm.isna().sum()
missing_pct = ((missing_counts / len(df_post_norm)) * 100).round(2)
missing_report = pd.DataFrame(
    {
        "Column": missing_counts.index,
        "Missing_Count": missing_counts.values,
        "Missing_Percent": missing_pct.values,
    }
).sort_values(["Missing_Count", "Missing_Percent"], ascending=False)

print("POST-CLEANING ANALYSIS")
print("=" * 70)
print(f"Rows: {rows:,}")
print(f"Columns: {cols}")
print(f"Duplicate rows: {duplicate_rows:,}")

print("\nMissing values report:")
print(missing_report.to_string(index=False))

if "AGE" in df_post_norm.columns:
    age_stats = df_post_norm["AGE"].describe().round(2)
    print("\nAGE summary:")
    print(age_stats.to_string())

if "CITY" in df_post_norm.columns:
    city_non_missing = int(df_post_norm["CITY"].notna().sum())
    city_coverage_pct = (city_non_missing / len(df_post_norm)) * 100
    print(f"\nCITY coverage: {city_non_missing:,}/{len(df_post_norm):,} ({city_coverage_pct:.2f}%)")

    print("Top 10 CITY values:")
    print(df_post_norm["CITY"].value_counts().head(10).to_string())

if "EMAIL" in df_post_norm.columns:
    domains = (
        df_post_norm["EMAIL"]
        .dropna()
        .astype(str)
        .str.extract(r"@([A-Za-z0-9.-]+\.[A-Za-z]{2,})", expand=False)
        .dropna()
        .str.lower()
    )

    top_domains = domains.value_counts().head(10)
    print("\nTop 10 email domains:")
    print(top_domains.to_string())
else:
    top_domains = pd.Series(dtype=int)

post_missing_file = PROCESSED_DIR / "new_meet_post_cleaning_missing_report.csv"
missing_report.to_csv(post_missing_file, index=False, encoding="utf-8")

post_domains_file = PROCESSED_DIR / "new_meet_post_cleaning_top_email_domains.csv"
(
    top_domains.rename_axis("Domain")
    .reset_index(name="Count")
    .to_csv(post_domains_file, index=False, encoding="utf-8")
)

post_summary_file = PROCESSED_DIR / "new_meet_post_cleaning_summary.csv"
pd.DataFrame(
    [
        {
            "Rows": rows,
            "Columns": cols,
            "Duplicate_Rows": duplicate_rows,
            "Complete_Rows": int((~df_post_norm.isna().any(axis=1)).sum()),
        }
    ]
).to_csv(post_summary_file, index=False, encoding="utf-8")

print("\nSaved reports:")
print(f"- {post_missing_file}")
print(f"- {post_domains_file}")
print(f"- {post_summary_file}")

POST-CLEANING ANALYSIS
Rows: 133,340
Columns: 6
Duplicate rows: 0

Missing values report:
    Column  Missing_Count  Missing_Percent
  PASSWORD              3              0.0
  USERNAME              2              0.0
FIRST_NAME              0              0.0
       AGE              0              0.0
      CITY              0              0.0
     EMAIL              0              0.0

AGE summary:
count    133340.00
mean         32.44
std          10.28
min          18.00
25%          25.00
50%          31.00
75%          38.00
max          82.00

CITY coverage: 133,340/133,340 (100.00%)
Top 10 CITY values:
CITY
Abidjan       16747
Alger          9681
Yaoundé        7707
Yaounde        7608
Douala         6978
Tunis          5580
Casablanca     4643
Paris          4606
Rabat          3602
Bejaia         2211

Top 10 email domains:
EMAIL
yahoo.fr       46517
hotmail.fr     31993
live.fr        15843
hotmail.com    12671
gmail.com       8288
yahoo.com       6719
ymail.com       1397


#### Copy Cleaned Dataset to Ingested Folder
Transfer the final cleaned CSV to the ingested folder for downstream processing and archival.

In [20]:
# Define paths
processed_dir = Path(r"\\BCA-ORG-00\Users Folder\jalleyne\Desktop\Protexxa\data_cleaning_scripts\new meet_dataset\new meet_raw\new meet_processed")
ingested_dir = Path(r"\\BCA-ORG-00\Users Folder\jalleyne\Desktop\Protexxa\data_cleaning_scripts\new meet_dataset\new meet_raw\new meet_ingested")

cleaned_file = processed_dir / "new_meet_cleaned.csv"
ingested_file = ingested_dir / "new_meet_cleaned.csv"

# Ensure destination folder exists
ingested_dir.mkdir(parents=True, exist_ok=True)

# Copy file
if cleaned_file.exists():
    shutil.copy2(cleaned_file, ingested_file)
    print(f"✓ File copied successfully")
    print(f"  Source: {cleaned_file}")
    print(f"  Target: {ingested_file}")
    print(f"  File size: {ingested_file.stat().st_size:,} bytes")
else:
    print(f"✗ Source file not found: {cleaned_file}")

✓ File copied successfully
  Source: \\BCA-ORG-00\Users Folder\jalleyne\Desktop\Protexxa\data_cleaning_scripts\new meet_dataset\new meet_raw\new meet_processed\new_meet_cleaned.csv
  Target: \\BCA-ORG-00\Users Folder\jalleyne\Desktop\Protexxa\data_cleaning_scripts\new meet_dataset\new meet_raw\new meet_ingested\new_meet_cleaned.csv
  File size: 11,253,556 bytes
